In [2]:
import numpy as np
import scipy.stats as stats

f) Сгенерируйте выборку объема n = 100 для некоторого значения параметра $\theta$

In [59]:
n = 100
theta_true = 10
beta = 0.95
np.random.seed(46)

sample = np.random.uniform(theta_true, 2 * theta_true, n)
x_max = np.max(sample)
x_mean = np.mean(sample)

theta_omm = (2/3) * x_mean
theta_omp_corrected = ((n + 1) / (2 * n + 1)) * x_max


print(f"Выбранное theta: {theta_true}")
print(f"Выборка: {sample}")
print(f"Оценка метода моментов: {theta_omm:.4f}")
print(f"Оценка метода максимального правдоподобия (исправленная): {theta_omp_corrected:.4f}")

Выбранное theta: 10
Выборка: [17.83832351 16.34833706 12.49043092 17.58075865 13.13076936 19.37237357
 10.42865448 14.40867203 19.1272229  14.55003919 15.07934341 10.84907955
 14.26349791 17.4571696  18.67081396 13.23665827 11.04074397 18.02108047
 13.94831949 16.27768088 10.35988544 12.9990785  10.4765322  13.72211078
 12.62415196 19.90526206 13.95145348 13.07539263 12.21101136 14.94025025
 10.5787519  12.89700041 12.34950391 19.63430415 16.15673873 18.62583969
 10.90085256 11.48988902 16.72574964 19.78991925 10.23223505 19.07715691
 10.58051316 17.55665434 10.64602878 12.55733424 11.69607764 11.95127297
 11.06556851 15.207896   10.06246276 17.90605096 11.38710924 11.07145195
 14.29601164 12.18972877 18.35851729 10.0587472  11.48322895 16.98343843
 11.47609905 18.70784536 15.13986217 14.24800256 10.18968378 14.94325926
 19.46624129 13.45803428 14.07328893 18.28926431 17.14015397 14.86571771
 12.32744401 16.71996489 15.78654682 14.48385694 15.76512955 13.11097212
 16.52071362 10.909506

Вычислите указанные выше доверительные интервалы для доверительной вероятности 0.95

In [74]:
# d) Точный доверительный интервал
t1 = 1 + ((1 - beta) / 2) ** (1 / n)
t2 = 1 + ((1 + beta) / 2) ** (1 / n)

start = x_max / t2
end = x_max / t1
l =  end - start
print(f"Точный доверительный интервал: ({start:.4f}, {end:.4f}), l = {l:.4f}")

# e) Асимптотический доверительный интервал (на основе ОММ)
t1 = stats.norm.ppf((1 - beta) / 2)
t2 = stats.norm.ppf((1 + beta) / 2)
alpha1 = x_mean
alpha2 = 1 / n * np.sum(sample ** 2)
start_omm = theta_omm - 2 / 3 / (n ** 0.5) * (alpha2 - alpha1 ** 2) ** 0.5 * t2
end_omm = theta_omm - 2 / 3 / (n ** 0.5) * (alpha2 - alpha1 ** 2) ** 0.5 * t1
l_omm = end_omm - start_omm
print(f"Асимптотический доверительный интервал: ({start_omm:.4f}, {end_omm:.4f}), l = {l_omm:.4f}")

Точный доверительный интервал: (9.9539, 10.1362), l = 0.1823
Асимптотический доверительный интервал: (9.2281, 9.9790), l = 0.7509


g) Численно постройте бутстраповский доверительный интервал

In [78]:
# непараметрический ОММ
k = 1000
bootstrap_estimates = []
for _ in range(k):
    boot_sample = np.random.choice(sample, size=n, replace=True)
    boot_theta = 2 / 3 * np.mean(boot_sample)
    bootstrap_estimates.append(boot_theta - theta_omm)

bootstrap_estimates = np.array(sorted(bootstrap_estimates))
k1 = int((1 - beta) / 2 * k - 1)
k2 = int((1 + beta) / 2 * k - 1)

start_boot_omm = theta_omm - bootstrap_estimates[k2]
end_boot_omm = theta_omm - bootstrap_estimates[k1]
l_boot_omm = end_boot_omm - start_boot_omm
print(f"Непараметрический бутстраповский доверительный интервал (ОММ): ({start_boot_omm:.4f}, {end_boot_omm:.4f}), l = {l_boot_omm:.4f}")


Непараметрический бутстраповский доверительный интервал (ОММ): (9.2166, 9.9820), l = 0.7653


In [76]:
# непараметрический ОМП
k = 1000
bootstrap_estimates = []
for _ in range(k):
    boot_sample = np.random.choice(sample, size=n, replace=True)
    boot_theta = (n + 1) / (2 * n + 1) * np.max(boot_sample)
    bootstrap_estimates.append(boot_theta - theta_omp_corrected)

bootstrap_estimates = np.array(sorted(bootstrap_estimates))
k1 = int((1 - beta) / 2 * k - 1)
k2 = int((1 + beta) / 2 * k - 1)

start_boot_omp = theta_omm - bootstrap_estimates[k2]
end_boot_omp = theta_omm - bootstrap_estimates[k1]
l_boot_omp = end_boot_omp - start_boot_omp
print(f"Непараметрический бутстраповский доверительный интервал (ОМП): ({start_boot_omp:.4f}, {end_boot_omp:.4f}), l = {l_boot_omp:.4f}")

Непараметрический бутстраповский доверительный интервал (ОМП): (9.6036, 9.8242), l = 0.2206


h) Сравнить все интервалы

In [79]:
intervals = [("Точный", l),
             ("Ассимптотический (ОММ)", l_omm),
             ("Непараметрический бутстрап (ОММ)", l_boot_omm),
             ("Непараметрический бутстрап (ОМП)", l_boot_omp)]
intervals = sorted(intervals, key=lambda x: x[1])

for name, length in intervals:
    print(f"{name}: l = {length:.4f}")

Точный: l = 0.1823
Непараметрический бутстрап (ОМП): l = 0.2206
Ассимптотический (ОММ): l = 0.7509
Непараметрический бутстрап (ОММ): l = 0.7653
